In [77]:
%%configure -f
{"executorMemory": "12G", "executorCores": 12, "ttl": "12h", "heartbeatTimeoutInSecond": 43200, "numExecutors": 3}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
11,None,pyspark,idle,,,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
11,None,pyspark,idle,,,None,✔


In [78]:
import pyspark.sql.functions as F

results_gm12878_df = (
    spark
    .read
    .parquet("s3a://database/results/a1fc46a9-93f8-424f-b41d-37bfd85d3b94")
    .withColumn("cell_line", F.lit("GM12878"))
)

results_h1esc_df = (
    spark
    .read
    .parquet("s3a://database/results/f7bc6dac-6aa3-49e6-a2e5-c2ff27824c81")
    .withColumn("cell_line", F.lit("H1ESC"))
)

results_hffc6_df = (
    spark
    .read
    .parquet("s3a://database/results/f648f805-c3a9-4cf4-a108-94e6f5fa96c1")
    .withColumn("cell_line", F.lit("HFFC6"))
)

results = results_gm12878_df.union(results_h1esc_df).union(results_hffc6_df)

chromatin_states_df = (
    spark
    .read
    .parquet("s3a://database/chromatin_states")
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [79]:
active_gene_states = ['TssA', 'TssAFlnk', 'TxFlnk', 'Tx', 'TxWk', 'EnhG', 'EnhG1', 'EnhG2', 'Enh', 'EnhA1', 'EnhA2']
used_projects = ['whole_all_vs_all_gm12878_fix', 'whole_all_vs_all_h1esc_fix', 'whole_all_vs_all_hffc6_fix']

chromatin_states_df = (
    chromatin_states_df
    .where(F.col('name').isin(active_gene_states))
)

results = (
    results
    .where("avg_dist > 0 AND var_dist > 0")
    .where(F.col('project_id').isin(used_projects))
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [80]:
results.createOrReplaceTempView("results")
chromatin_states_df.createOrReplaceTempView("chromatin_states")

query = f"""
SELECT DISTINCT
    r.project_id,
    r.gene_id,
    r.enh_id,
    r.avg_dist,
    r.var_dist,
    r.cell_line
FROM results r
WHERE EXISTS (
    SELECT 1
    FROM chromatin_states cs_gene
    WHERE cs_gene.cell_line = r.cell_line
      AND cs_gene.chrom = r.gene_chr
      AND cs_gene.start <= r.gene_end
      AND cs_gene.end >= r.gene_start
)
AND EXISTS (
    SELECT 1
    FROM chromatin_states cs_enh
    WHERE cs_enh.cell_line = r.cell_line
      AND cs_enh.chrom = r.enh_chr
      AND cs_enh.start <= r.enh_end
      AND cs_enh.end >= r.enh_start
)
"""
results = spark.sql(query)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [81]:
results_by_gene = (
    results
    .groupBy('project_id', 'gene_id', 'cell_line')
    .agg(
        F.avg('avg_dist').alias('avg_dist'),
        F.min('avg_dist').alias('min_dist'),
        # F.expr("""
        # aggregate(
        #     slice(sort_array(collect_list(avg_dist)), 1, 3),
        #     cast(0 as float),
        #     (acc, x) -> acc + x
        # ) / size(slice(sort_array(collect_list(avg_dist)), 1, 3))
        # """).alias('min_dist'),
        F.max('avg_dist').alias('max_dist'),
    )
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
results_by_gene.repartition(1).write.mode('overwrite').parquet("s3a://database/test/closest_enh_distance_by_gene_v3")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…